In [1]:
import pandas as pd
import re
from pathlib import Path

FILE_PATH = Path(r"/home/yyyy/codework/GARplus/GNN/code/DDA_test/data/去病图数据/gene_disease.csv")
df = pd.read_csv(FILE_PATH, sep=None, engine="python")

print("数据形状:", df.shape)
print("表头字段:")
print(df.columns.tolist())

print("\n================ 缺失值比例 ================\n")
na_stats = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_ratio": df.isna().mean().round(4)
}).sort_values("missing_ratio", ascending=False)
print(na_stats)

# ---------- 1. DirectEvidence ----------
if "DirectEvidence" in df.columns:
    print("\n================ DirectEvidence 分布 ================\n")
    print(df["DirectEvidence"].value_counts(dropna=False))
    print("\n不同取值:")
    print(df["DirectEvidence"].dropna().astype(str).unique())

# ---------- 2. InferenceScore ----------
if "InferenceScore" in df.columns:
    print("\n================ InferenceScore 分布 ================\n")
    score = pd.to_numeric(df["InferenceScore"], errors="coerce")
    print(score.describe())

    bins = [-float("inf"), 0, 1, 5, 10, 20, 50, 100, float("inf")]
    labels = ["<=0", "0-1", "1-5", "5-10", "10-20", "20-50", "50-100", ">100"]
    bucket = pd.cut(score, bins=bins, labels=labels)
    print("\n分箱统计:")
    print(bucket.value_counts(dropna=False).sort_index())

# ---------- 3. DiseaseName 关键词 ----------
if "DiseaseName" in df.columns:
    print("\n================ DiseaseName 负向关键词统计 ================\n")
    negative_keywords = [
        "drug-induced", "toxicity", "poisoning", "overdose",
        "dependence", "withdrawal", "adverse", "injury",
        "damage", "failure", "necrosis"
    ]
    text = df["DiseaseName"].fillna("").astype(str).str.lower()
    for kw in negative_keywords:
        cnt = text.str.contains(kw, regex=False).sum()
        print(f"{kw}: {cnt}")

# ---------- 4. 多值字段长度分布 ----------
def split_count(x):
    if pd.isna(x) or str(x).strip() == "":
        return 0
    parts = re.split(r"[|;,]", str(x))
    parts = [p.strip() for p in parts if p.strip()]
    return len(parts)

for col in ["InferenceGeneSymbol", "OmimIDs", "PubMedIDs"]:
    if col in df.columns:
        counts = df[col].apply(split_count)
        print(f"\n================ {col} 多值长度分布 ================\n")
        print(counts.value_counts().sort_index().head(20))

数据形状: (877940, 14)
表头字段:
['Node_1', 'Node_2', 'EdgeLabel', 'GeneSymbol', 'GeneID', 'DiseaseName', 'DiseaseID', 'DirectEvidence', 'InferenceChemicalName', 'InferenceScore', 'OmimIDs', 'PubMedIDs', 'gene_index', 'disease_index']

================ 缺失值比例 ================

                       missing_count  missing_ratio
OmimIDs                       872567         0.9939
DirectEvidence                845266         0.9628
InferenceChemicalName         294065         0.3349
InferenceScore                294065         0.3349
PubMedIDs                     265843         0.3028
GeneSymbol                    261391         0.2977
GeneID                        261391         0.2977
DiseaseName                   261391         0.2977
DiseaseID                     261391         0.2977
gene_index                    261391         0.2977
Node_1                             0         0.0000
Node_2                             0         0.0000
EdgeLabel                          0         0.0000
dis

In [12]:
import pandas as pd
from pathlib import Path

FILE_PATH = Path(r"/home/yyyy/codework/GARplus/GNN/code/DDA_test/data/去病图数据/gene_disease.csv")
df = pd.read_csv(FILE_PATH, sep=None, engine="python")

# ===== 1. 负语义关键词（根据你的统计优化） =====
negative_keywords = [
    "drug-induced",
    "toxicity", "poisoning", "overdose",
    "injury", "necrosis", "failure",
    "adverse", "withdrawal", "dependence"
]

def is_negative_disease(name):
    name = str(name).lower()
    return any(k in name for k in negative_keywords)

# ===== 2. 分类函数 =====
def classify_edge(row):

    direct = str(row.get("DirectEvidence", "") or "").lower()
    disease = str(row.get("DiseaseName", "") or "").lower()
    score = row.get("InferenceScore")

    is_pos = False
    is_neg = False
    both_no = True

    reasons_pos = []
    reasons_neg = []

    # ===== 1. positive =====
    if direct == "therapeutic":
        is_pos = True
        reasons_pos.append("direct:therapeutic")

    # ===== 2. negative =====
    if direct != "therapeutic":

        # ---- 2.1 疾病负语义（副作用类）----
        # for k in negative_keywords:
        #     if k in disease:
        #         is_neg = True
        #         reasons_neg.append(f"disease:{k}")
        #         break

        # # ---- 2.2 机制关系 ----
        if direct == "marker/mechanism" and is_negative_disease(disease):
            both_no = True
            is_neg = True
            reasons_neg.append("direct:mechanism")

        # ---- 2.3 高推理强度 ----
        if pd.notna(score) and score > 50:
            both_no = True
            # is_neg = True
            # reasons_neg.append("inference:high_score")

    # ===== 3. 决策 =====
    if is_pos and not is_neg:
        return "positive", ";".join(reasons_pos)

    elif is_neg and not is_pos:
        return "negative", ";".join(reasons_neg)

    elif is_pos and is_neg:
        return "neutral_conflict", f"POS[{';'.join(reasons_pos)}]|NEG[{';'.join(reasons_neg)}]"

    else:
        return "neutral", ""


# ===== 3. 应用 =====
labels, reasons = zip(*df.apply(classify_edge, axis=1))

df["edge_semantic"] = labels
df["edge_reason"] = reasons

# ===== 4. 统计 =====
print(df["edge_semantic"].value_counts())

# ===== 5. 导出 =====
pos_neg = df[df["edge_semantic"].isin(["positive", "negative"])].copy()
pos_neg["label"] = (
    (pos_neg["edge_semantic"] == "positive") 
    # |
    # (pos_neg["edge_semantic"] == "neutral")
).astype(int)

# pos_neg.to_csv(FILE_PATH.with_name("drug_disease_pos_neg.csv"), index=False, encoding="utf-8-sig")

# ===== 6. 查看负边样例 =====
print("\n===== 负边样例 =====")
print(
    df[df["edge_semantic"]=="negative"][
        ["DiseaseName","DirectEvidence","InferenceScore","edge_reason"]
    ].head(20).to_string(index=False)
)

edge_semantic
neutral     874811
positive      1769
negative      1360
Name: count, dtype: int64

===== 负边样例 =====
                                    DiseaseName   DirectEvidence  InferenceScore      edge_reason
                           Liver Failure, Acute marker/mechanism             NaN direct:mechanism
                              Arsenic Poisoning marker/mechanism             NaN direct:mechanism
                            Acute Kidney Injury marker/mechanism             NaN direct:mechanism
                                  Heart Failure marker/mechanism             NaN direct:mechanism
                  Substance Withdrawal Syndrome marker/mechanism             NaN direct:mechanism
                                  Heart Failure marker/mechanism             NaN direct:mechanism
                            Acute Kidney Injury marker/mechanism             NaN direct:mechanism
Drug-Related Side Effects and Adverse Reactions marker/mechanism             NaN direct:mechanism
   

In [13]:
import numpy as np
SAVE_DIR = Path("/home/yyyy/codework/GARplus/GNN/code/TI_test/data_signed")
results = df.apply(classify_edge, axis=1)
df["edge_semantic"] = results.str[0]
df["edge_reason"] = results.str[1]

print("===== edge_semantic 分布 =====")
print(df["edge_semantic"].value_counts(dropna=False))

# 可选：保存完整语义文件
semantic_path = SAVE_DIR / "gene_disease_with_semantic.csv"
df.to_csv(semantic_path, index=False, encoding="utf-8-sig")
print(f"\n已保存语义标注文件: {semantic_path}")

# ===== 4. 只保留 positive / negative =====
pos_neg_df = df[df["edge_semantic"].isin(["positive", "negative"])].copy()
pos_neg_df["label"] = pos_neg_df["edge_semantic"].map({
    "positive": 1,
    "negative": 2
}).astype(int)

print("\n===== 正负边统计 =====")
print("positive:", (pos_neg_df["label"] == 1).sum())
print("negative:", (pos_neg_df["label"] == 2).sum())
print("total:", len(pos_neg_df))

# ===== 5. 重新给节点连续编号，保证 node/edge 对齐 =====
required_cols = ["gene_index", "disease_index"]
for c in required_cols:
    if c not in pos_neg_df.columns:
        raise ValueError(f"缺少必要字段: {c}")

pos_neg_df = pos_neg_df.rename(columns={
    "gene_index": "index_A",
    "disease_index": "index_B"
})

all_nodes = pd.unique(
    pd.concat(
        [pos_neg_df["index_A"], pos_neg_df["index_B"]],
        ignore_index=True
    )
)

id_map = pd.Series(
    index=all_nodes,
    data=range(len(all_nodes))
)

pos_neg_df["src"] = pos_neg_df["index_A"].map(id_map)
pos_neg_df["dst"] = pos_neg_df["index_B"].map(id_map)

assert not pos_neg_df["src"].isna().any()
assert not pos_neg_df["dst"].isna().any()

pos_neg_df["src"] = pos_neg_df["src"].astype(int)
pos_neg_df["dst"] = pos_neg_df["dst"].astype(int)

print("\n重新编号后节点总数:", len(all_nodes))

# ===== 6. 保存 node 文件 =====
nodes_df = pd.DataFrame({
    "node_id": id_map.values,
    "old_index": id_map.index
}).sort_values("node_id").reset_index(drop=True)

nodes_path = SAVE_DIR / "node_labeled.csv"
nodes_df.to_csv(nodes_path, index=False, encoding="utf-8-sig")
print(f"保存节点文件: {nodes_path}")

# ===== 7. 保存完整边文件（带语义）=====
edges_full = pos_neg_df[["src", "dst", "label", "edge_semantic", "edge_reason"]].copy()
edges_full["rel"] = "drug_disease"

edges_full_path = SAVE_DIR / "edges_labeled_with_reason.csv"
edges_full[["src", "dst", "rel", "label", "edge_semantic", "edge_reason"]].to_csv(
    edges_full_path, index=False, encoding="utf-8-sig"
)
print(f"保存带语义原因的边文件: {edges_full_path}")

# ===== 8. 构建正边池、负边池 =====
df_pos_all = edges_full[edges_full["label"] == 1][["src", "dst", "label"]].drop_duplicates().reset_index(drop=True)
df_neg_all = edges_full[edges_full["label"] == 2][["src", "dst", "label"]].drop_duplicates().reset_index(drop=True)

print("\n===== 去重后的正负边池 =====")
print("正边池:", len(df_pos_all))
print("负边池:", len(df_neg_all))

# ===== 9. 固定正边，负边逐渐增加 =====
seed = 42
np.random.seed(seed)

# 这里定义你想要的正负比例：Pos:Neg = r:1
# r 越大，负边越少；r 越小，负边越多
ratios = [60, 50, 40, 30, 20, 10]

total_neg_available = len(df_neg_all)

# 为了保证所有数据集使用同一份正边，固定正边数
fixed_pos_num = len(df_pos_all)
pos_fixed_sample = df_pos_all.sample(n=fixed_pos_num, replace=False, random_state=seed).copy()

print("\n===== 固定正边设置 =====")
print("固定正边数量:", len(pos_fixed_sample))
print("负边池总量:", total_neg_available)

# 按“负边数量逐渐增加”排序
ratio_plan = []
for r in ratios:
    neg_need = int(fixed_pos_num / r)
    neg_need = min(neg_need, total_neg_available)
    ratio_plan.append((r, neg_need))

# 去重，避免不同 ratio 算出相同 neg_need
ratio_plan = list(dict.fromkeys(ratio_plan))
ratio_plan = sorted(ratio_plan, key=lambda x: x[1])

ratio_dir = SAVE_DIR / "ratio_datasets"
ratio_dir.mkdir(parents=True, exist_ok=True)

summary = []

for r, neg_need in ratio_plan:
    neg_sample = df_neg_all.sample(n=neg_need, replace=False, random_state=seed).copy()

    cur_edges = pd.concat([pos_fixed_sample, neg_sample], axis=0)
    cur_edges = cur_edges.sample(frac=1.0, random_state=seed).reset_index(drop=True)
    cur_edges["rel"] = "drug_disease"

    filename = f"edges_ratio{r}x.csv"
    edges_path = ratio_dir / filename

    cur_edges[["src", "dst", "rel", "label"]].to_csv(edges_path, index=False, encoding="utf-8-sig")

    print(f"[生成完成] Ratio {r}x | Pos: {len(pos_fixed_sample)} | Neg: {neg_need} | Total: {len(cur_edges)}")

    summary.append({
        "ratio_label": f"{r}x",
        "imbalance_ratio": f"{r}:1",
        "pos_count": len(pos_fixed_sample),
        "neg_count": neg_need,
        "file_name": filename
    })

# ===== 10. 保存汇总 =====
summary_df = pd.DataFrame(summary)
summary_path = ratio_dir / "dataset_summary.csv"
summary_df.to_csv(summary_path, index=False, encoding="utf-8-sig")

print("\n===== 数据集统计汇总 =====")
print(summary_df)
print(f"\n保存汇总文件: {summary_path}")

===== edge_semantic 分布 =====
edge_semantic
neutral     874811
positive      1769
negative      1360
Name: count, dtype: int64

已保存语义标注文件: /home/yyyy/codework/GARplus/GNN/code/TI_test/data_signed/gene_disease_with_semantic.csv

===== 正负边统计 =====
positive: 1769
negative: 1360
total: 3129

重新编号后节点总数: 2063
保存节点文件: /home/yyyy/codework/GARplus/GNN/code/TI_test/data_signed/node_labeled.csv
保存带语义原因的边文件: /home/yyyy/codework/GARplus/GNN/code/TI_test/data_signed/edges_labeled_with_reason.csv

===== 去重后的正负边池 =====
正边池: 1769
负边池: 1360

===== 固定正边设置 =====
固定正边数量: 1769
负边池总量: 1360
[生成完成] Ratio 60x | Pos: 1769 | Neg: 29 | Total: 1798
[生成完成] Ratio 50x | Pos: 1769 | Neg: 35 | Total: 1804
[生成完成] Ratio 40x | Pos: 1769 | Neg: 44 | Total: 1813
[生成完成] Ratio 30x | Pos: 1769 | Neg: 58 | Total: 1827
[生成完成] Ratio 20x | Pos: 1769 | Neg: 88 | Total: 1857
[生成完成] Ratio 10x | Pos: 1769 | Neg: 176 | Total: 1945

===== 数据集统计汇总 =====
  ratio_label imbalance_ratio  pos_count  neg_count           file_name
0         60x  